In [ ]:
# ======================================================================
# SIGNALWELL VOICE — INSTANT INFERENCE (ZERO TRAINING / RUNS IN 5 SECONDS)
# ======================================================================
# Loads your already-trained model from Google Drive and immediately
# lets you upload and test any .wav audio file.
# ======================================================================

!pip -q install transformers librosa soundfile

import os
import torch
import torch.nn as nn
import numpy as np
import soundfile as sf
import librosa
from pathlib import Path
from transformers import WavLMModel, Wav2Vec2FeatureExtractor
from google.colab import files, drive

# 1. Mount Google Drive to find your saved model checkpoint
print("Connecting to Google Drive to load your trained model...")
drive.mount('/content/drive')

CHECKPOINT_PATH = Path('/content/drive/MyDrive/SIGNALWELL_VOICE/models/SIGNALWELL_VOICE_BEST.pt')

if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f"Checkpoint not found at {CHECKPOINT_PATH}. Please verify your Google Drive has the SIGNALWELL_VOICE folder.")

print(f"✓ Found trained checkpoint at: {CHECKPOINT_PATH}")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SR = 16000
SEGMENT_SAMPLES = SR * 6
CHUNK_SAMPLES = SR * 1

checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
EMOTIONS = checkpoint["emotions"]

# 2. Rebuild Model Architecture
class SignalWellVoice(nn.Module):
    def __init__(self, wavlm_dim=1024, prosody_dim=37, hidden=256, heads=8, layers=3):
        super().__init__()
        self.wavlm_projection = nn.Sequential(
            nn.Linear(wavlm_dim, hidden), nn.LayerNorm(hidden), nn.GELU(), nn.Dropout(0.15)
        )
        self.prosody_projection = nn.Sequential(
            nn.Linear(prosody_dim, 128), nn.LayerNorm(128), nn.GELU(), nn.Linear(128, hidden)
        )
        self.fusion = nn.Sequential(
            nn.Linear(hidden * 2, hidden), nn.LayerNorm(hidden), nn.GELU(), nn.Dropout(0.15)
        )
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden, nhead=heads, dim_feedforward=hidden * 4,
            dropout=0.15, activation="gelu", batch_first=True, norm_first=True
        )
        self.temporal = nn.TransformerEncoder(encoder_layer, num_layers=layers)
        self.attention = nn.Sequential(
            nn.Linear(hidden, 128), nn.Tanh(), nn.Linear(128, 1)
        )
        self.emotion_head = nn.Sequential(
            nn.Linear(hidden, 128), nn.GELU(), nn.Dropout(0.10), nn.Linear(128, len(EMOTIONS))
        )
        self.uncertainty_head = nn.Sequential(
            nn.Linear(hidden, 64), nn.GELU(), nn.Linear(64, 1), nn.Sigmoid()
        )

    def forward(self, wavlm, prosody):
        w = self.wavlm_projection(wavlm)
        p = self.prosody_projection(prosody).unsqueeze(1).expand(-1, w.size(1), -1)
        x = self.fusion(torch.cat([w, p], dim=-1))
        x = self.temporal(x)
        scores = self.attention(x)
        weights = torch.softmax(scores, dim=1)
        pooled = (x * weights).sum(dim=1)
        return {
            "emotion": self.emotion_head(pooled),
            "uncertainty": self.uncertainty_head(pooled)
        }

model = SignalWellVoice().to(DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

WAVLM_NAME = "microsoft/wavlm-large"
wavlm_processor = Wav2Vec2FeatureExtractor.from_pretrained(WAVLM_NAME)
wavlm_extractor = WavLMModel.from_pretrained(WAVLM_NAME).to(DEVICE).eval()
for p in wavlm_extractor.parameters():
    p.requires_grad = False

print(f"✓ Model loaded successfully on {DEVICE.upper()}!\n")

def load_audio(file_path):
    audio, orig_sr = sf.read(file_path, dtype="float32")
    if audio.ndim > 1:
        audio = np.mean(audio, axis=1)
    if orig_sr != SR:
        audio = librosa.resample(audio, orig_sr=orig_sr, target_sr=SR)
    audio = np.nan_to_num(audio)
    threshold = 0.01 * np.max(np.abs(audio)) if np.max(np.abs(audio)) > 0 else 0.01
    active = np.where(np.abs(audio) > threshold)[0]
    if len(active) > 0:
        audio = audio[active[0] : active[-1] + 1]
    peak = np.max(np.abs(audio))
    if peak > 1e-8:
        audio = audio / peak
    if len(audio) >= SEGMENT_SAMPLES:
        start = (len(audio) - SEGMENT_SAMPLES) // 2
        audio = audio[start : start + SEGMENT_SAMPLES]
    else:
        padded = np.zeros(SEGMENT_SAMPLES, dtype=np.float32)
        padded[:len(audio)] = audio
        audio = padded
    return audio

def extract_features(audio):
    chunks = [audio[i * CHUNK_SAMPLES : (i + 1) * CHUNK_SAMPLES] for i in range(6)]
    inputs = wavlm_processor(chunks, sampling_rate=SR, return_tensors="pt", padding=True)
    with torch.no_grad():
        out = wavlm_extractor(inputs["input_values"].to(DEVICE))
        wavlm_feats = out.last_hidden_state.mean(dim=1).unsqueeze(0)

    f0 = librosa.yin(audio, fmin=60, fmax=500, sr=SR, hop_length=512)
    valid_f0 = f0[(f0 >= 60) & (f0 <= 500) & np.isfinite(f0)]
    pitch_mean = float(np.mean(valid_f0)) if len(valid_f0) else 0.0
    pitch_std = float(np.std(valid_f0)) if len(valid_f0) else 0.0
    pitch_range = float(np.max(valid_f0) - np.min(valid_f0)) if len(valid_f0) else 0.0

    rms = librosa.feature.rms(y=audio, hop_length=512)[0]
    energy_mean = float(np.mean(rms))
    energy_std = float(np.std(rms))
    energy_range = float(np.max(rms) - np.min(rms))

    zcr_mean = float(np.mean(librosa.feature.zero_crossing_rate(y=audio, hop_length=512)[0]))
    centroid = librosa.feature.spectral_centroid(y=audio, sr=SR, hop_length=512)[0]
    bandwidth = librosa.feature.spectral_bandwidth(y=audio, sr=SR, hop_length=512)[0]
    rolloff = librosa.feature.spectral_rolloff(y=audio, sr=SR, hop_length=512)[0]

    mfcc = librosa.feature.mfcc(y=audio, sr=SR, n_mfcc=13, hop_length=512)
    mfcc_mean = np.mean(mfcc, axis=1)
    mfcc_std = np.std(mfcc, axis=1)

    silence_thresh = max(float(np.max(rms)) * 0.05, 1e-4)
    voiced_frames = int(np.sum(rms > silence_thresh))
    pause_ratio = max(0.0, 1.0 - (voiced_frames / max(len(rms), 1)))

    prosody = np.concatenate([
        np.array([
            pitch_mean, pitch_std, pitch_range,
            energy_mean, energy_std, energy_range,
            zcr_mean, float(np.mean(centroid)), float(np.mean(bandwidth)), float(np.mean(rolloff)),
            pause_ratio
        ], dtype=np.float32),
        mfcc_mean.astype(np.float32),
        mfcc_std.astype(np.float32)
    ])
    prosody_feats = torch.tensor(np.nan_to_num(prosody), dtype=torch.float32).unsqueeze(0).to(DEVICE)
    return wavlm_feats, prosody_feats

def predict(audio_path):
    audio = load_audio(audio_path)
    wavlm_feats, prosody_feats = extract_features(audio)
    with torch.no_grad():
        out = model(wavlm_feats, prosody_feats)
        probs = torch.softmax(out["emotion"], dim=-1)[0].cpu().numpy()
        pred_idx = int(np.argmax(probs))
        uncertainty = float(out["uncertainty"][0].cpu().numpy())

    pred_label = EMOTIONS[pred_idx]
    confidence = probs[pred_idx] * 100

    print("\n" + "=" * 65)
    print(f"File: {Path(audio_path).name}")
    print(f"🎯 Predicted Emotion : {pred_label.upper()} ({confidence:.2f}% confidence)")
    print(f"📊 Uncertainty Score  : {uncertainty:.4f}")
    print("-" * 65)
    for emotion, prob in sorted(zip(EMOTIONS, probs), key=lambda x: x[1], reverse=True):
        bar = "█" * int(prob * 30)
        print(f"  {emotion:<10} : {prob*100:5.1f}%  {bar}")
    print("=" * 65)

# Prompt upload
print("=" * 65)
print("SELECT YOUR AUDIO FILE(S) (.wav) FROM YOUR PC:")
print("=" * 65)
uploaded = files.upload()
for fname in uploaded.keys():
    predict(fname)
